# MedPercept-Bench — Colab notebook (memory-safe, resumable)

Zero-shot evaluation of the frozen **WeVisOne-VL** model on small medical datasets (Kvasir-SEG, BCCD).

**Before running:** Runtime → Change runtime type → **T4 GPU**. Put this project folder in your Google Drive
at `MyDrive/MedPercept-Bench`. If a cell fails, fix it and re-run from that cell: inference **resumes**
where it stopped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. GPU check + mount Drive

In [ ]:
import os, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.used','--format=csv'],capture_output=True,text=True).stdout)

DRIVE_PROJECT = '/content/drive/MyDrive/MedPercept-Bench'   # where you uploaded the folder
WORK = '/content/MedPercept-Bench'                          # fast local copy (Drive is slow for many small files)
assert os.path.isdir(DRIVE_PROJECT), f'Folder not found: {DRIVE_PROJECT}'

name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 0 MiB



## 2. Install the light way

Upstream `requirements.txt` pins `torch==2.11`, `numpy==2.2.6` and **`vllm`**. Installing that on Colab downloads many GB
and replaces Colab's torch, which can crash the session. We install only the small packages and WeVisOne-VL with `--no-deps`.

In [ ]:
%%bash
set -e
pip install -q "transformers==5.12.1" accelerate qwen_vl_utils pycocotools supervision huggingface_hub pyyaml pandas
if [ ! -d /content/WeVisOne-VL ]; then git clone -q https://github.com/Tencent/WeVisOne-VL.git /content/WeVisOne-VL; fi
pip install -q --no-deps -e /content/WeVisOne-VL
python -c "import wevisone_vl; print('wevisone_vl OK', wevisone_vl.__version__)"


wevisone_vl OK 1.0.0


## 3. Copy project to local disk + download datasets

In [ ]:
!rsync -a --exclude data --exclude .git "$DRIVE_PROJECT"/ "$WORK"/
os.chdir(WORK)
!bash scripts/download_data.sh

[1/2] Kvasir-SEG ...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
curl: (60) SSL certificate problem: unable to get local issuer certificate
More details here: https://curl.se/docs/sslcerts.html

curl failed to verify the legitimacy of the server and therefore could not
establish a secure connection to it. To learn more about this situation and
how to fix it, please visit the web page mentioned above.
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 44.0M  100 44.0M    0     0  7777k      0  0:00:05  0:00:05 --:--:--  9.9M
[2/2] BCCD ...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left 

## 4. Choose the run size

| preset | Kvasir imgs | BCCD imgs | rough time on T4 |
|---|---|---|---|
| `smoke` | 5 | 5 | a few minutes (check everything works) |
| `quick` | 30 | 30 | 10–20 min (start here) |
| `standard` | 100 | 100 | ~1 h |

Go `smoke` -> `quick` -> `standard`. Times are estimates, not measured.

In [9]:
PRESET = 'quick'
MODEL  = 'tencent/WeVisOne-VL-2B'
SIZES  = {'smoke': 5, 'quick': 30, 'standard': 100}

import yaml
cfg = yaml.safe_load(open('configs/datasets.yaml'))
cfg['model']['name'] = MODEL
cfg['model']['device'] = 'cuda:0'
cfg['kvasir']['subset'] = SIZES[PRESET]
cfg['bccd']['subset']   = SIZES[PRESET]
yaml.safe_dump(cfg, open('configs/datasets.yaml','w'))
print('preset:', PRESET, SIZES[PRESET], 'images per dataset')

preset: quick 30 images per dataset


## 5. Run inference

Protections: images capped at ~0.8 MP, answers capped at 1024 tokens, GPU cache freed every 5 images, an OOM on one image is
logged and skipped, and every image is saved as soon as it finishes. **If the session dies, re-run this cell** (it resumes).
Use `--fresh` to start over, e.g. after changing preset or settings.

In [10]:
!python scripts/run_inference.py --config configs/datasets.yaml

Loading model tencent/WeVisOne-VL-2B on device=cuda:0 ...
[ERROR] `loss` is part of Qwen3VLCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_vl/modeling_qwen3_vl.py.
[ERROR] `logits` is part of Qwen3VLCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_vl/modeling_qwen3_vl.py.
Loading weights: 100% 626/626 [00:00<00:00, 2598.84it/s]
Kvasir-SEG: 30 images
    [kvasir_seg_polyp] 10/30
    [kvasir_seg_polyp] 20/30
    [kvasir_seg_polyp] 30/30
  [kvasir_seg_polyp] 30 images (0 resumed, 0 OOM) -> /content/MedPercept-Bench/results/predictions/kvasir_seg_polyp.json
    [kvasir_detection] 10/30
    [kvasir_detection] 20/30
    [kvasir_detection] 30/30
  [kvasir_detection] 30 images (0 resumed, 0 OOM) -> /content/MedPercept-Ben

## 6. Evaluate + figures

In [ ]:
!python scripts/evaluate.py --config configs/datasets.yaml
!python scripts/make_gallery.py --n 13
!python scripts/make_readme_figures.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/figures/*.png'))[:8]:
    print(f); display(Image(f))

## 7. Save results back to Drive (before closing Colab)

In [ ]:
!rsync -a results/ "$DRIVE_PROJECT"/results/
print('Saved to', DRIVE_PROJECT + '/results')